In [1]:
import os

In [2]:
%pwd

'd:\\Text_Summarize_Project\\Text_Summarizer_Project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Text_Summarize_Project\\Text_Summarizer_Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir:Path
    data_path:Path
    model_ckpt:Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [7]:
from text_summarize.constants import *
from text_summarize.utils.common import read_yaml,create_directories

In [8]:
class ConfigurationManager:
    def __init__(self,config_file = CONFIG_FILE_PATH,param_file = PARAM_FILE_PATH):
        self.config = read_yaml(config_file)
        self.params = read_yaml(param_file)

    def get_model_training_object(self)->ModelTrainingConfig:
        config = self.config.model_training
        params = self.params.TrainingArguments
        model_trainer_config = ModelTrainingConfig(
                root_dir=config.root_dir,
                data_path=config.data_path,
                model_ckpt = config.model_ckpt,
                num_train_epochs = params.num_train_epochs,
                warmup_steps = params.warmup_steps,
                per_device_train_batch_size = params.per_device_train_batch_size,
                weight_decay = params.weight_decay,
                logging_steps = params.logging_steps,
                evaluation_strategy = params.evaluation_strategy,
                eval_steps = params.evaluation_strategy,
                save_steps = params.save_steps,
                gradient_accumulation_steps = params.gradient_accumulation_steps
            )
        return model_trainer_config


In [9]:
import torch
from datasets import load_dataset, load_from_disk
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments, Trainer

c:\Users\LENOVO\anaconda3\envs\textSum\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class ModelTraining:
    def __init__(self,config:ModelTrainingConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)
        dataset_samsum_pt = load_from_disk(self.config.data_path)
        trainer_args = TrainingArguments(
            output_dir='pegasus-samsum',num_train_epochs=1,warmup_steps=500,
            per_device_train_batch_size=1,per_device_eval_batch_size=1,
            weight_decay=0.01,logging_steps=10,eval_strategy='steps',
            eval_steps=500,save_steps=1e6,gradient_accumulation_steps=16)
        trainer = Trainer(model=model_pegasus,args=trainer_args,
                  data_collator=seq2seq_data_collator,train_dataset=dataset_samsum_pt['test'],
                  eval_dataset=dataset_samsum_pt['validation'])
        trainer.train()
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))


In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_training_object()
    model_trainer_config = ModelTraining(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-04-12 11:17:21,031:INFO:common:yaml file config\config.yaml loaded successfuly]
[2026-04-12 11:17:21,063:INFO:common:yaml file params.yaml loaded successfuly]


[2026-04-12 11:17:21,546:INFO:_client:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-12 11:17:21,580:INFO:_client:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-04-12 11:17:21,920:INFO:_client:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-12 11:17:21,953:INFO:_client:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-04-12 11:17:22,272:INFO:_client:HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-04-12 11:17:22

[2026-04-12 11:17:29,289:WARNING:_http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-04-12 11:17:30,228:INFO:_client:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/refs%2Fpr%2F12/model.safetensors "HTTP/1.1 302 Found"]


Loading weights: 100%|██████████| 680/680 [00:02<00:00, 274.85it/s]
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-04-12 11:18:12,510:INFO:_client:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-12 11:18:13,199:INFO:_client:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]


c:\Users\LENOVO\anaconda3\envs\textSum\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
